## DPU Data Reuse MCP - evaluation run
Sends every question in `test_cases.csv` to the LIVE MCP app, checks the answer, times it, and saves results to `results/`.

**Before the first run:** the app `dpu-reuse-mcp` must give **Inspector Gadget** the **Can use** permission (app page > Permissions).

**How to run:** click **Run all**. Read the summary table at the end. Nothing in the app or the data is changed.

In [0]:
%pip install -q --upgrade databricks-mcp databricks-sdk nest_asyncio
%restart_python

In [0]:
APP_NAME = "dpu-reuse-mcp"
MCP_URL = "https://dpu-reuse-mcp-7474654188777746.aws.databricksapps.com/mcp"
EVAL_DIR = "/Workspace/Users/ying.huang@nih.gov/Data-Product-Usability/mcp/eval"
RUN_NOTE = "24 cases incl. find_studies (post 2026-11-13 update)"

In [0]:
import json
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

nest_asyncio.apply()

# Inspector Gadget = shared test identity; its sign-in is kept in a UC secret, not in this notebook.
ig = json.loads(dbutils.secrets.get(catalog="models", schema="ai", key="inspector-gadget"))
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
ig_client = WorkspaceClient(host=host, client_id=ig["client_id"], client_secret=ig["client_secret"])
mcp_client = DatabricksMCPClient(server_url=MCP_URL, workspace_client=ig_client)

# Which code version is live (for the record)
deployment_id = WorkspaceClient().apps.get(APP_NAME).active_deployment.deployment_id
print("Connected. Live deployment:", deployment_id)
print("Tools:", [t.name for t in mcp_client.list_tools()])

In [0]:
import time
import pandas as pd

cases = pd.read_csv(f"{EVAL_DIR}/test_cases.csv", dtype=str, keep_default_na=False)


def get_path(obj, path):
    """Follow a dotted path like results.0.summary.reuse_papers."""
    if not path:
        return obj
    for part in path.split("."):
        if isinstance(obj, list):
            obj = obj[int(part)] if part.isdigit() and int(part) < len(obj) else None
        elif isinstance(obj, dict):
            obj = obj.get(part)
        else:
            return None
        if obj is None:
            return None
    return obj


def check(case, ok, payload, err):
    """Return (passed, what_we_got)."""
    mt, exp = case["match_type"], case["expected"]
    if mt == "tool_error":
        return (not ok), ("rejected: " + err[:120]) if not ok else "NOT rejected"
    if not ok:
        return False, "call failed: " + err[:200]
    got = get_path(payload, case["check_path"])
    if mt == "equals":
        return str(got) == exp, str(got)[:200]
    if mt == "contains":
        return exp in str(got), str(got)[:200]
    if mt == "list_has_pmid":
        pmids = [str(p.get("pmid")) for p in (got or [])]
        return exp in pmids, f"{len(pmids)} papers listed"
    return False, "unknown match_type " + mt


def call(case):
    """Call the live server. Returns (ok, payload, error_text)."""
    if case["tool"] == "__list_tools__":
        return True, ";".join(sorted(t.name for t in mcp_client.list_tools())), ""
    r = mcp_client.call_tool(case["tool"], json.loads(case["arguments_json"]))
    text = r.content[0].text if r.content else ""
    if getattr(r, "isError", False):
        return False, None, text
    return True, json.loads(text), ""


rows = []
for case in cases.to_dict("records"):
    t0 = time.perf_counter()
    try:
        ok, payload, err = call(case)
    except Exception as e:  # network or validation error
        ok, payload, err = False, None, str(e)
    passed, got = check(case, ok, payload, err)
    rows.append({
        "case_id": case["case_id"], "category": case["category"],
        "question_plain": case["question_plain"], "expected": case["expected"],
        "got": got, "passed": passed, "seconds": round(time.perf_counter() - t0, 3),
    })

results = pd.DataFrame(rows)
display(results)

In [0]:
run_time = pd.Timestamp.now(tz="America/New_York").strftime("%Y-%m-%d %H:%M")
by_cat = results.groupby("category").agg(cases=("passed", "size"), passed=("passed", "sum")).reset_index()
by_cat["pass_rate_%"] = (100 * by_cat["passed"] / by_cat["cases"]).round(1)

summary = pd.DataFrame([{
    "run_time": run_time,
    "deployment_id": deployment_id,
    "cases": len(results),
    "passed": int(results["passed"].sum()),
    "pass_rate_%": round(100 * results["passed"].mean(), 1),
    "median_seconds": round(results["seconds"].median(), 3),
    "p95_seconds": round(results["seconds"].quantile(0.95), 3),
    "note": RUN_NOTE,
}])
print("How to read: pass_rate = % of test questions answered as expected; seconds = time per call (includes network).")
display(summary)
display(by_cat)
failures = results[~results["passed"]]
if len(failures):
    display(failures)
else:
    print("No failures: all cases passed.")

In [0]:
import os

os.makedirs(f"{EVAL_DIR}/results", exist_ok=True)
stamp = run_time.replace(" ", "_").replace(":", "")
results.assign(run_time=run_time, deployment_id=deployment_id).to_csv(
    f"{EVAL_DIR}/results/run_{stamp}.csv", index=False)

log_path = f"{EVAL_DIR}/results/summary_log.csv"
summary.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("Saved:", f"results/run_{stamp}.csv", "and one line in results/summary_log.csv")